# Baseline — XGBoost (MovieLens)
OHE + multi-hot · Default hyperparameters
**Saves to**: `baseline_XGB\`

## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Numerical preprocessing
STEP 4 — Build OHE feature matrix
STEP 5 — Train & evaluate
STEP 6 — Feature importance
STEP 7 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from xgboost               import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute        import SimpleImputer
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from scipy.sparse          import csr_matrix, hstack

In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
OUT_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\baseline_XGB"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Saving results to: {{OUT_DIR}}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

TARGET    = 'rating'
CAT_GENRE = 'genres'
CAT_LANG  = 'original_language'
NUM_COLS  = [
    'imdbRating', 'imdbVotes', 'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values
print(f"Train: {{df_train.shape}}  Val: {{df_val.shape}}  Test: {{df_test.shape}}")


---
## Step 3 — Numerical Preprocessing

In [ ]:
# Tag features: missing = no tag activity → 0
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] = \
        df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)


In [ ]:
# Median imputation — fit on train only
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])


In [ ]:
# log1p transform
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))


In [ ]:
# StandardScaler — fit on train only
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])
print("Numerical preprocessing done.")


---
## Step 4 — Build OHE Feature Matrix

In [ ]:
# Multi-hot encode genres — fit on train, reindex val/test
genres_tr = df_train[CAT_GENRE].fillna('Unknown').str.get_dummies('|')
genre_cols = genres_tr.columns
genres_v  = df_val[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)
genres_te = df_test[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)
print(f"Genre columns: {{len(genre_cols)}}")


In [ ]:
# OHE language — fit on train only
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
lang_tr = ohe.fit_transform(df_train[[CAT_LANG]].fillna('unknown'))
lang_v  = ohe.transform(df_val[[CAT_LANG]].fillna('unknown'))
lang_te = ohe.transform(df_test[[CAT_LANG]].fillna('unknown'))
print(f"Language columns: {{len(ohe.categories_[0])}}")


In [ ]:
# Stack numeric + language + genre into sparse matrix
X_train = hstack([csr_matrix(df_train[NUM_COLS].values), lang_tr, csr_matrix(genres_tr.values)])
X_val   = hstack([csr_matrix(df_val[NUM_COLS].values),   lang_v,  csr_matrix(genres_v.values)])
X_test  = hstack([csr_matrix(df_test[NUM_COLS].values),  lang_te, csr_matrix(genres_te.values)])
feat_names = NUM_COLS + ohe.get_feature_names_out([CAT_LANG]).tolist() + list(genre_cols)
print(f"Feature matrix — train: {{X_train.shape}}")


---
## Step 5 — Train & Evaluate
`tree_method='hist'` for speed — not a tunable hyperparameter.

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae


In [ ]:
t0 = time.perf_counter()
model = XGBRegressor(tree_method='hist', random_state=42, verbosity=0)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
train_time = time.perf_counter() - t0

tr_rmse,  tr_mae  = get_metrics(y_train, model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  model.predict(X_test))

print('=' * 50)
print('XGBoost — Baseline Results')
print('=' * 50)
print(f'  Train      RMSE: {tr_rmse:.4f}   MAE: {tr_mae:.4f}')
print(f'  Validation RMSE: {val_rmse:.4f}   MAE: {val_mae:.4f}')
print(f'  Test       RMSE: {te_rmse:.4f}   MAE: {te_mae:.4f}')
print(f'  Train time : {train_time:.2f}s')
print(f'  Train-Test gap : {te_rmse - tr_rmse:.4f}')

---
## Step 6 — Feature Importance
Gain — average loss reduction per split.

In [ ]:
gain_scores = model.get_booster().get_score(importance_type='gain')
feat_map    = {f'f{i}': name for i, name in enumerate(feat_names)}
imp_df = (pd.DataFrame({'feature': list(gain_scores.keys()), 'gain': list(gain_scores.values())})
            .assign(feature=lambda x: x['feature'].map(feat_map).fillna(x['feature']))
            .sort_values('gain', ascending=False).head(15))
plt.figure(figsize=(8,4))
plt.barh(imp_df['feature'][::-1], imp_df['gain'][::-1], color='steelblue')
plt.xlabel('Gain'); plt.title('XGBoost Baseline — Top 15 Features')
plt.tight_layout(); plt.show()

---
## Step 7 — Save Results

In [ ]:
result = {
    'model'        : 'XGBoost (Baseline)',
    'train_rmse'   : round(tr_rmse,  4),
    'val_rmse'     : round(val_rmse, 4),
    'test_rmse'    : round(te_rmse,  4),
    'train_mae'    : round(tr_mae,   4),
    'val_mae'      : round(val_mae,  4),
    'test_mae'     : round(te_mae,   4),
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'xgb_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'xgb_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Results saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse:.4f}  Test MAE: {te_mae:.4f}")
